# 🔬 Phase 3A: Validation Sweep — Tìm Config Cosine Decay Tối Ưu

**Mục tiêu:** Thử 10+ cấu hình `(α, K, decay)` trên 50 mẫu validation  
**Tìm ra:** Config cosine decay tốt nhất để dùng cho Phase 3B và 3C  
**Thời gian ước tính:** ~2 giờ  
**Input:** Phase 1 artifacts + Dataset 15K  
**Output:** `phase3a_val_sweep.json` chứa best config

---

In [ ]:
# Cell 1: Install
!pip install -q bitsandbytes accelerate transformers torch rouge-score tqdm
print('✅ Done')

In [ ]:
# Cell 2: Imports & Seed
import os, json, glob, random, time, math, gc
import numpy as np
import torch
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

OUTPUT_DIR = '/kaggle/working'
print(f'GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
# Cell 3: Data Loading (IDENTICAL split to Phase 1/2)
DATA_FILENAME = 'vietnamese_medical_halueval_15k_specialized.json'
search_paths = [
    f'/kaggle/input/**/{DATA_FILENAME}',
    f'/kaggle/input/{DATA_FILENAME}',
    f'data/{DATA_FILENAME}',
    f'./{DATA_FILENAME}'
]
data_path = None
for pattern in search_paths:
    matches = glob.glob(pattern, recursive=True)
    if matches:
        data_path = matches[0]
        break
if not data_path:
    raise FileNotFoundError(f'❌ {DATA_FILENAME} not found')

with open(data_path, 'r', encoding='utf-8') as f:
    raw_dataset = json.load(f)

shuffled_records = list(raw_dataset)
random.seed(SEED)
random.shuffle(shuffled_records)

n_total = len(shuffled_records)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)
val_records = shuffled_records[n_train:n_train + n_val]
test_records = shuffled_records[n_train + n_val:]

print(f'📊 Total: {n_total:,} | Val: {len(val_records):,} | Test: {len(test_records):,}')

In [ ]:
# Cell 4: Load Phase 1 Artifacts
config_paths = glob.glob('/kaggle/input/**/steering_config.json', recursive=True)
v_steer_paths = glob.glob('/kaggle/input/**/v_steer.pt', recursive=True)
v_rand_paths = glob.glob('/kaggle/input/**/v_rand.pt', recursive=True)

if not config_paths:
    raise FileNotFoundError('❌ steering_config.json not found')

with open(config_paths[0], 'r') as f:
    steering_config = json.load(f)

BEST_LAYER = steering_config['best_layer']
v_steer = torch.load(v_steer_paths[0], map_location='cpu')
v_rand = torch.load(v_rand_paths[0], map_location='cpu')

print(f'✅ Layer: {BEST_LAYER} | v_steer: {v_steer.shape}')

In [ ]:
# Cell 5: Load Model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
print(f'⌛ Loading {MODEL_NAME}...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map='auto', trust_remote_code=True
)
model.eval()
print('✅ Model loaded!')

In [ ]:
# Cell 6: Hook + Evaluation Engine
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

PROMPT_TEMPLATE = """Dựa vào ngữ cảnh y học sau đây, hãy trả lời câu hỏi:
Ngữ cảnh: {context}
Câu hỏi: {question}
Trả lời: """

class SteeringHook:
    def __init__(self, layer_idx, v_vector, alpha=20.0, K=8, decay='hard'):
        self.layer_idx = layer_idx
        self.v_vector = v_vector
        self.alpha = alpha
        self.K = K
        self.decay = decay
        self.step_counter = 0
        self.handle = None
    
    def _eff_alpha(self, t):
        if self.K >= 999: return self.alpha
        if t >= self.K: return 0.0
        if self.decay == 'hard': return self.alpha
        elif self.decay == 'cosine':
            return self.alpha * 0.5 * (1.0 + math.cos(math.pi * t / self.K))
        elif self.decay == 'linear':
            return self.alpha * (1.0 - t / self.K)
        return self.alpha
    
    def hook_fn(self, module, inputs, output):
        a = self._eff_alpha(self.step_counter)
        if a > 0:
            if isinstance(output, tuple):
                h = output[0]
                v = self.v_vector.to(h.device).to(h.dtype)
                h[:, -1, :] = h[:, -1, :] + a * v
                output = (h,) + output[1:]
            else:
                v = self.v_vector.to(output.device).to(output.dtype)
                output[:, -1, :] = output[:, -1, :] + a * v
        self.step_counter += 1
        return output
    
    def register(self, mdl):
        self.step_counter = 0
        self.handle = mdl.model.layers[self.layer_idx].register_forward_hook(self.hook_fn)
    
    def remove(self):
        if self.handle: self.handle.remove(); self.handle = None


def evaluate_condition(model, tokenizer, records, v_vector, alpha, K, decay,
                       max_new_tokens=80, name='', layer_idx=None):
    results = []
    t_start = time.time()
    for idx, rec in enumerate(tqdm(records, desc=name)):
        ctx = rec.get('knowledge_context', rec.get('context', ''))
        q = rec['question']
        ref = rec['right_answer']
        prompt = PROMPT_TEMPLATE.format(context=ctx, question=q)
        inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
        
        hook = None
        if v_vector is not None and layer_idx is not None:
            hook = SteeringHook(layer_idx, v_vector, alpha=alpha, K=K, decay=decay)
            hook.register(model)
        
        torch.manual_seed(SEED + idx)
        t0 = time.time()
        with torch.no_grad():
            out_ids = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                     do_sample=True, temperature=0.1, top_p=0.85)
        elapsed_ms = (time.time() - t0) * 1000
        if hook: hook.remove()
        
        gen_tokens = out_ids[0][inputs.input_ids.shape[1]:]
        gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True)
        r_score = scorer.score(ref, gen_text)['rougeL'].fmeasure * 100
        
        eos_id = tokenizer.eos_token_id
        hit_eos = bool(len(gen_tokens) > 0 and gen_tokens[-1].item() == eos_id)
        
        # 4-gram repetition
        words = gen_text.split()
        if len(words) >= 4:
            ngrams = [tuple(words[i:i+4]) for i in range(len(words)-3)]
            rep4 = 1.0 - len(set(ngrams))/len(ngrams) if ngrams else 0.0
        else:
            rep4 = 0.0
        
        results.append({
            'idx': idx, 'rouge_l': r_score, 'num_tokens': len(gen_tokens),
            'elapsed_ms': elapsed_ms, 'hit_eos': hit_eos, 'rep_4gram': rep4,
            'generated': gen_text, 'reference': ref, 'question': q,
            'category': rec.get('hallucination_type', 'unknown'),
        })
    
    avg_rl = np.mean([r['rouge_l'] for r in results])
    total_min = (time.time() - t_start) / 60
    print(f'  ✅ [{name}] {total_min:.1f}min | ROUGE-L: {avg_rl:.2f}%')
    return results

print('✅ Engine ready.')
# Verify cosine schedule
h = SteeringHook(0, v_steer, alpha=20.0, K=8, decay='cosine')
print('Cosine schedule (α=20,K=8):', [f'{h._eff_alpha(t):.1f}' for t in range(12)])

In [ ]:
# Cell 7: ═══ RUN VALIDATION SWEEP ═══
print('='*70)
print('VALIDATION SWEEP: 50 samples × 12 configs')
print('='*70)

val_subset = val_records[:50]

# --- Baseline ---
print('\n--- Vanilla Baseline ---')
val_bl = evaluate_condition(model, tokenizer, val_subset,
    v_vector=None, alpha=0, K=0, decay='hard', name='Baseline')
val_bl_rl = np.mean([r['rouge_l'] for r in val_bl])

# --- Full Steering (α=20, K=∞) ---
print('\n--- Full Steering (α=20, K=∞) ---')
val_fs = evaluate_condition(model, tokenizer, val_subset,
    v_vector=v_steer, alpha=20.0, K=999, decay='hard',
    name='Full Steer', layer_idx=BEST_LAYER)
val_fs_rl = np.mean([r['rouge_l'] for r in val_fs])

# --- Cosine Decay configs ---
configs = [
    {'alpha': 8.0,  'K': 8,  'decay': 'cosine'},
    {'alpha': 12.0, 'K': 8,  'decay': 'cosine'},
    {'alpha': 15.0, 'K': 8,  'decay': 'cosine'},
    {'alpha': 18.0, 'K': 8,  'decay': 'cosine'},
    {'alpha': 8.0,  'K': 16, 'decay': 'cosine'},
    {'alpha': 12.0, 'K': 16, 'decay': 'cosine'},
    {'alpha': 15.0, 'K': 16, 'decay': 'cosine'},
    {'alpha': 18.0, 'K': 16, 'decay': 'cosine'},
    {'alpha': 15.0, 'K': 16, 'decay': 'linear'},
    {'alpha': 18.0, 'K': 16, 'decay': 'linear'},
]

sweep = {}
for cfg in configs:
    tag = f"{cfg['decay']}_a{cfg['alpha']}_K{cfg['K']}"
    res = evaluate_condition(model, tokenizer, val_subset,
        v_vector=v_steer, alpha=cfg['alpha'], K=cfg['K'], decay=cfg['decay'],
        name=tag, layer_idx=BEST_LAYER)
    sweep[tag] = {'config': cfg, 'rouge_l': np.mean([r['rouge_l'] for r in res])}

gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 8: ═══ RESULTS TABLE ═══
print('\n' + '='*70)
print('📊 VALIDATION SWEEP RESULTS')
print('='*70)
print(f'{"Config":<30} {"ROUGE-L":>10} {"vs BL":>8} {"vs Full":>8}')
print('-'*60)
print(f'{"Vanilla Baseline":<30} {val_bl_rl:>9.2f}% {"---":>8} {"---":>8}')
print(f'{"Full Steering (a=20,K=inf)":<30} {val_fs_rl:>9.2f}% {val_fs_rl-val_bl_rl:>+7.2f} {"---":>8}')
print('-'*60)

ranked = sorted(sweep.items(), key=lambda x: x[1]['rouge_l'], reverse=True)
for name, info in ranked:
    rl = info['rouge_l']
    d_bl = rl - val_bl_rl
    d_fs = rl - val_fs_rl
    s_bl = f'+{d_bl:.2f}' if d_bl >= 0 else f'{d_bl:.2f}'
    s_fs = f'+{d_fs:.2f}' if d_fs >= 0 else f'{d_fs:.2f}'
    star = ' ⭐' if d_bl > 0 and d_fs > 0 else (' ✓' if d_bl > 0 else '')
    print(f'{name:<30} {rl:>9.2f}% {s_bl:>8} {s_fs:>8}{star}')

best_name = ranked[0][0]
best_cfg = ranked[0][1]['config']
runner_name = ranked[1][0]
runner_cfg = ranked[1][1]['config']

print(f'\n{"="*70}')
print(f'🏆 BEST CONFIG: {best_name}')
print(f'   alpha={best_cfg["alpha"]}, K={best_cfg["K"]}, decay={best_cfg["decay"]}')
print(f'🥈 RUNNER-UP:   {runner_name}')
print(f'   alpha={runner_cfg["alpha"]}, K={runner_cfg["K"]}, decay={runner_cfg["decay"]}')
print(f'\n⚠️  GHI LẠI CÁC GIÁ TRỊ NÀY ĐỂ DÙNG CHO NOTEBOOK 3B VÀ 3C!')
print(f'={"="*70}')

In [ ]:
# Cell 9: Save Results
export = {
    'baseline_rouge_l': val_bl_rl,
    'full_steering_rouge_l': val_fs_rl,
    'sweep': {n: {'config': i['config'], 'rouge_l': i['rouge_l']} for n, i in sweep.items()},
    'best_config': best_cfg,
    'runner_up_config': runner_cfg,
    'best_name': best_name,
    'runner_up_name': runner_name,
}
with open(os.path.join(OUTPUT_DIR, 'phase3a_val_sweep.json'), 'w') as f:
    json.dump(export, f, indent=2)

print('💾 Saved: phase3a_val_sweep.json')
print('\n🎉 PHASE 3A HOÀN THÀNH!')
print(f'\nBước tiếp: Chạy Notebook 3B với config:')
print(f'  BEST_ES_ALPHA = {best_cfg["alpha"]}')
print(f'  BEST_ES_K = {best_cfg["K"]}')
print(f'  BEST_ES_DECAY = "{best_cfg["decay"]}"')